# Crossref API Query Template

This notebook provides a template for querying the Crosref works endpoint for metadata. API query parameter names and values are entered as dictionary keys and values., e.g.

`params = {'rows': '100'}`

Similarly, query filters are entered in a separate dictionary:

`filters = {'type': 'journal-article'}`

To use the notebook:
1. Make a copy (go to 'File' and 'Save a copy to drive')
1. Fill out the details of the notebook in the text cell immediately after this one.
1. Add any analysis functions you'll use in the section 'Functions for querying and analysis'
1. In the 'Query the API' section modify the query parameters to the ones you'll need.
1. Run the query, then run your analysis functions in the final section.

Send any feeback on this template to Martyn. Thanks!



# Notebook Title

This notebook... [what does this notebook do?].

It does the following:

1. Define functions for querying and analysis.
1. Retrieve [short description of data queried for].
1. Determine/calculate [description of any analysis procedures on the output].

## 1. Functions for querying and analysis

In [ ]:

import requests # library for API querying
import json     # library for saving dictionaries to json
import pprint   # library for nice display of output

In [ ]:
# some useful functions and variables
works_api_url = 'https://api.crossref.org/v1/works'

def get_works(params:dict, filters = {}) -> tuple[dict, int]:
    '''
    Query the Crossref works endpoint and return json. In case of a problem returning
    the results, an empty dictionary is returned.

    See https://api.crossref.org for API documentation and information on available parameters

    parameters
    ----------
    params: dict
        query parameters for the Crossref works API, e.g. {'rows': 1, 'mailto':'myemail@demo.org'}
    filters: dict
        query filters e.g. {'from-pub-date': '2024-01-01', 'type': 'journal-article'}

    returns:
    dict:
        json output of the API as a dictionary
        the list of metadata records is in d['message']['items']
    int:
        the html status code of the query, e.g.
        200 for a successful query
        4XX for a user error
        5XX for a server error
    '''

    # reformat and add filters to the query parameters if used
    if len(list(filters.keys())) > 0:
        params['filter'] = filters_to_params(filters)

    # make the query
    r = requests.get(works_api_url, params=params)

    # get the json output from the API response
    if r.status_code == 200:
        js = r.json()
    else:
        # something went wrong, print the response and return an empty dictionary
        print(r.text)
        js = {}
    return js, r.status_code


def filters_to_params(filters:dict) -> str:
    ''' Set the filters parameter for the Crossref works API using a dictionary

    parameters
    ----------
    filters: dict
        filters for the Crossref works endpoint, e.g. {'from-pub-date': '2024-01-01', 'type': 'journal-article'}

    returns
    -------
    str:
        the filters expressed as a string, e.g. 'from-pub-date:2024-01-01,type:journal-article'
        This should be added to the query parameters dictionary with the key 'filter'

    '''

    # create an empty string for the filter values
    fval = ''
    # add each filter key and value to the string
    for f in filters:
        fval += str(f) + ':' + str(filters[f]) + ','
    fval = fval[:-1] # removing trailing comma

    return fval


def save_to_json(data:dict, fname:str) -> None:
    ''' save a dictionary as a json file '''
    with open(fname, 'w') as f:
        json.dump(data, f, indent=2)

In [ ]:
# analysis functions
def get_total_results(js):
    ''' print the number of total results for the query made '''

    try:
        print(f"{js['message']['total-results']} metadata records match the query\n{len(js['message']['items'])} retrieved")
    except KeyError:
        print(f"Results could not be extracted from the query, did it run correctly?")

def print_records(js, n):
  ''' print the first n records from a dictionary containing json output from the Crossref works endpoint. '''

  try:
    d = js['message']['items'][:n]
  except KeyError:
    print('JSON format incorrect, items not found')
  pprint.pprint(d)

# 2. Query the API

Customize the parameters and filters then run the query. See https://api.crossref.org for API documentation.

In [ ]:
# enter query parameters and filters
params = {
    'rows': 1,
    'mailto': 'demo@crossref.org',
    'select' : ['DOI'] # use this if you only need a small number of fields in the output
}
filters = {
    'type': 'journal-article',
    'has-references': 1
}

In [ ]:
# make the query
js, status = get_works(params, filters)
print(f'Query ran with status {status}')

# 3. Analysis

Customize this section to perform any analysis you would like on the query output.

In [ ]:
get_total_results(js)

In [ ]:
print_records(js, 5)

In [ ]:
save_to_json(js, 'test.json')